# DPO Rewriter Training

Stage 2 of Simurgh's two-stage training: Direct Preference Optimization of the Qwen3-4B query rewriter using preference pairs generated by `notebooks/gen_dpo_data.ipynb`. The trained LoRA adapter teaches the rewriter to reformulate exam questions in a way that matches each learner persona.

**Kaggle setup checklist**
1. Enable GPU accelerator (T4 x1 is sufficient; ~15–30 min for 3 epochs).
2. Enable internet access (model downloaded from Hugging Face).
3. Attach the `simurgh-data` dataset — must contain the `dpo/` subdirectory with `train.jsonl` and `val.jsonl`.
4. No API secrets needed — training makes no LLM calls.

In [ ]:
!pip install -q unsloth

## Config

In [ ]:
import os

# Must be set before torch initializes CUDA; mitigates fragmentation OOMs.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
# Change DATASET_SLUG to match the name you gave your Kaggle dataset.
DATASET_SLUG = "simurgh-data"
DATA_ROOT = f"/kaggle/input/datasets/alirezahsn/{DATASET_SLUG}"
OUTPUT_DIR = "/kaggle/working/dpo_rewriter_checkpoints"

# ── Inline config (mirrors configs/phase4_dpo_rewriter.yaml) ─────────────────
CFG = {
    "data": {
        "train_path": f"{DATA_ROOT}/dpo/train.jsonl",
        "val_path": f"{DATA_ROOT}/dpo/val.jsonl",
    },
    "model": "Qwen/Qwen3-4B",
    "lora": {
        "r": 16,
        "alpha": 32,
        "dropout": 0.0,
        "target_modules": ["q_proj", "v_proj"],
    },
    "training": {
        "device": "cuda",
        "epochs": 10,
        "batch_size": 2,
        "gradient_accumulation_steps": 4,
        "lr": 5e-5,
        "beta": 0.1,
        "max_length": 256,
        "max_prompt_length": 200,
    },
    "checkpoint_dir": OUTPUT_DIR,
    "seed": 42,
}

Path(CFG["checkpoint_dir"]).mkdir(parents=True, exist_ok=True)

## Helpers and personas

In [ ]:
import json
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)


# ── Personas (from src/personalization/profiles.py) ───────────────────────────
PERSONAS = {
    "crammer": (
        "A ninth-grader who finds the textbook hard to follow and has little background "
        "on this topic. Mainly wants to pass the exam — give the answer and what is needed "
        "to score — but it must be spelled out simply, step by step, with examples."
    ),
    "scholar": (
        "A ninth-grader who reads dense material easily and has solid background on this "
        "topic. Wants to understand the underlying why and how, and the connections between "
        "ideas. Prefers a terse, high-level treatment without hand-holding or padding."
    ),
    "steady": (
        "A capable ninth-grader with average background on this topic. Wants a correct "
        "answer with a brief justification, balanced toward exam needs. Does not need "
        "elaborate scaffolding, but does appreciate a one-line reason."
    ),
}


def render_profile(persona_id: str) -> str:
    if persona_id not in PERSONAS:
        raise ValueError(f"Unknown persona: {persona_id!r}. Valid: {list(PERSONAS)}")
    return PERSONAS[persona_id]


# ── Rewrite prompt builder (from src/rag/rewriter.py) ────────────────────────
_REWRITE_SYSTEM = (
    "You are a query rewriting assistant for a Persian educational RAG system. "
    "Given a learner profile and an original question, rewrite the question as a "
    "retrieval query that will surface the most pedagogically useful passages for "
    "that specific learner. "
    "Rules: output ONLY the rewritten query — no explanation, no preamble, no quotes. "
    "Keep it in Persian if the original is Persian. "
    "You may expand abbreviations, add prerequisite terms, or rephrase for clarity, "
    "but do not invent facts or change the question's intent."
)


def _build_rewrite_prompt(profile_rendered: str, query: str) -> list:
    user = (
        f"Learner profile: {profile_rendered}\n\n"
        f"Original question: {query}\n\n"
        "Rewritten retrieval query:"
    )
    return [
        {"role": "system", "content": _REWRITE_SYSTEM},
        {"role": "user", "content": user},
    ]


# ── Data loading (from src/rl/dpo_train.py) ───────────────────────────────────
def load_pairs(path) -> list:
    pairs = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if line.strip():
            pairs.append(json.loads(line))
    return pairs


def pairs_to_dataset(pairs: list, tokenizer=None):
    from datasets import Dataset

    rows = []
    for p in pairs:
        msgs = _build_rewrite_prompt(render_profile(p["persona_id"]), p["query"])
        if tokenizer is not None:
            prompt = tokenizer.apply_chat_template(
                msgs,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        else:
            prompt = "\n\n".join(m["content"] for m in msgs)
        rows.append({"prompt": prompt, "chosen": p["chosen"], "rejected": p["rejected"]})
    return Dataset.from_list(rows)

## Health check

In [ ]:
train_pairs = load_pairs(CFG["data"]["train_path"])
val_pairs = load_pairs(CFG["data"]["val_path"])
logger.info("Train pairs: %d | Val pairs: %d", len(train_pairs), len(val_pairs))

train_dataset = pairs_to_dataset(train_pairs)
val_dataset = pairs_to_dataset(val_pairs)

assert set(train_dataset.column_names) == {"prompt", "chosen", "rejected"}, \
    f"Unexpected columns: {train_dataset.column_names}"

sample = train_pairs[0]
print(f"Sample persona : {sample['persona_id']}")
print(f"Sample query   : {sample['query'][:120]}")
print(f"Chosen  ({len(sample['chosen'])} chars): {sample['chosen'][:120]}")
print(f"Rejected({len(sample['rejected'])} chars): {sample['rejected'][:120]}")

## Training

In [ ]:
import torch
from unsloth import FastModel
from trl import DPOConfig, DPOTrainer

seed = CFG["seed"]
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = CFG["training"]["device"]
model_name = CFG["model"]
checkpoint_dir = Path(CFG["checkpoint_dir"])
train_cfg = CFG["training"]
lora_cfg = CFG["lora"]

logger.info("Loading tokenizer and model via Unsloth: %s", model_name)
model, tokenizer = FastModel.from_pretrained(
    model_name=model_name,
    dtype=None,           # auto: fp16 on T4, bf16 where supported
    max_seq_length=train_cfg["max_length"],
    load_in_4bit=True,    # NF4 quantization; ~2 GB vs ~8 GB for fp16
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Rebuild datasets with tokenizer: applies Qwen3's chat template with enable_thinking=False
# so the model never emits <think>…</think> tokens during DPO training.
train_dataset = pairs_to_dataset(train_pairs, tokenizer)
val_dataset = pairs_to_dataset(val_pairs, tokenizer)

model = FastModel.get_peft_model(
    model,
    r=lora_cfg["r"],
    lora_alpha=lora_cfg["alpha"],
    lora_dropout=lora_cfg["dropout"],
    target_modules=lora_cfg["target_modules"],
    bias="none",
    use_gradient_checkpointing="unsloth",
)
model.print_trainable_parameters()

training_args = DPOConfig(
    output_dir=str(checkpoint_dir),
    num_train_epochs=train_cfg["epochs"],
    per_device_train_batch_size=train_cfg["batch_size"],
    gradient_accumulation_steps=train_cfg["gradient_accumulation_steps"],
    learning_rate=train_cfg["lr"],
    beta=train_cfg["beta"],
    max_length=train_cfg["max_length"],
    max_prompt_length=train_cfg["max_prompt_length"],
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    seed=seed,
    fp16=device == "cuda",
    gradient_checkpointing=True,
    logging_steps=10,
    report_to="none",
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

logger.info("Starting DPO training...")
trainer.train()

best_dir = checkpoint_dir / "dpo_best"
trainer.save_model(str(best_dir))
logger.info("Best adapter saved to %s", best_dir)

training_log = {
    "config": CFG,
    "seed": seed,
    "log_history": trainer.state.log_history,
}
(checkpoint_dir / "training_log.json").write_text(
    json.dumps(training_log, ensure_ascii=False, indent=2), encoding="utf-8"
)
logger.info("Training log written to %s", checkpoint_dir / "training_log.json")

## Download

In [ ]:
!cd /kaggle/working && zip -r dpo_adapter.zip dpo_rewriter_checkpoints/dpo_best dpo_rewriter_checkpoints/training_log.json